# Demo-Run Local Footprint Visualization

Interactive visualization of the footprint field written to `outputs/demo-run-local/footprints.zarr`. Built on Plotly throughout for reliable rendering in VS Code's Jupyter and for hover/zoom/animation interrogation.

The footprint store now persists spatial and time-bin coordinates directly. This notebook reads those coordinates when available and falls back to reconstructing them from `run_metadata.json` for older runs.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import xarray as xr
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_DIR = PROJECT_ROOT / "outputs" / "demo-run-local"
FOOTPRINT_ZARR = RUN_DIR / "footprints.zarr"
RUN_METADATA_JSON = RUN_DIR / "run_metadata.json"

print("Project root:", PROJECT_ROOT)
print("Run dir:", RUN_DIR)
print("Footprint store exists:", FOOTPRINT_ZARR.exists())
print("Run metadata exists:", RUN_METADATA_JSON.exists())

Project root: /Users/chxmr/code/glide
Run dir: /Users/chxmr/code/glide/outputs/demo-run-local
Footprint store exists: True
Run metadata exists: True


## Load the footprint store and coordinates

This cell prefers persisted coordinates from `footprints.zarr` and only rebuilds them from `run_metadata.json` when opening an older store.

In [2]:
with RUN_METADATA_JSON.open("r", encoding="utf-8") as handle:
    run_metadata = json.load(handle)

footprint_ds = xr.open_zarr(FOOTPRINT_ZARR).load()
footprint = footprint_ds["footprint"]
if "y" in footprint.dims or "x" in footprint.dims:
    footprint = footprint.rename({"y": "latitude", "x": "longitude"})

config = run_metadata["config"]
release_lon = float(footprint_ds.attrs.get("release_lon", config["release_lon"]))
release_lat = float(footprint_ds.attrs.get("release_lat", config["release_lat"]))

n_time, n_z, n_y, n_x = footprint.shape


def _edges_from_centers(centers: np.ndarray) -> np.ndarray:
    centers = np.asarray(centers, dtype=float)
    if centers.size == 1:
        return np.array([centers[0] - 0.5, centers[0] + 0.5], dtype=float)

    deltas = np.diff(centers)
    edges = np.empty(centers.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (centers[:-1] + centers[1:])
    edges[0] = centers[0] - 0.5 * deltas[0]
    edges[-1] = centers[-1] + 0.5 * deltas[-1]
    return edges


has_persisted_coords = all(
    name in footprint_ds.coords
    for name in (
        "longitude",
        "latitude",
        "z_bin",
        "time_ago_start_hours",
        "time_ago_end_hours",
        "z_bottom_m",
        "z_top_m",
    )
)

if has_persisted_coords:
    if "time_ago" not in footprint.coords:
        footprint = footprint.assign_coords(time_ago=("time_ago", np.arange(n_time, dtype=int)))

    lon_centers = np.asarray(footprint.coords["longitude"].values, dtype=float)
    lat_centers = np.asarray(footprint.coords["latitude"].values, dtype=float)
    z_centers_m = np.asarray(footprint.coords["z_bin"].values, dtype=float)
    lon_edges = np.asarray(
        footprint_ds.coords.get("longitude_edge", _edges_from_centers(lon_centers)).values
        if "longitude_edge" in footprint_ds.coords
        else _edges_from_centers(lon_centers),
        dtype=float,
    )
    lat_edges = np.asarray(
        footprint_ds.coords.get("latitude_edge", _edges_from_centers(lat_centers)).values
        if "latitude_edge" in footprint_ds.coords
        else _edges_from_centers(lat_centers),
        dtype=float,
    )
    z_edges_m = np.concatenate(
        [
            np.asarray(footprint_ds.coords["z_bottom_m"].values, dtype=float),
            np.asarray([footprint_ds.coords["z_top_m"].values[-1]], dtype=float),
        ]
    )
    time_bin_start_h = np.asarray(footprint_ds.coords["time_ago_start_hours"].values, dtype=float)
    time_bin_end_h = np.asarray(footprint_ds.coords["time_ago_end_hours"].values, dtype=float)
    footprint = footprint.assign_coords(
        longitude=("longitude", lon_centers),
        latitude=("latitude", lat_centers),
        z_bin=("z_bin", z_centers_m),
    )
else:
    lon_min = release_lon - float(config["bbox_pad_lon_deg"])
    lon_max = release_lon + float(config["bbox_pad_lon_deg"])
    lat_min = release_lat - float(config["bbox_pad_lat_deg"])
    lat_max = release_lat + float(config["bbox_pad_lat_deg"])

    lon_edges = np.linspace(lon_min, lon_max, n_x + 1)
    lat_edges = np.linspace(lat_min, lat_max, n_y + 1)
    z_edges_m = np.linspace(0.0, 5000.0, n_z + 1)

    lon_centers = 0.5 * (lon_edges[:-1] + lon_edges[1:])
    lat_centers = 0.5 * (lat_edges[:-1] + lat_edges[1:])
    z_centers_m = 0.5 * (z_edges_m[:-1] + z_edges_m[1:])

    hours_per_bin = max(1.0, float(config["simulation_length_seconds"]) / 3600.0 / max(1, n_time))
    time_bin_start_h = np.arange(n_time, dtype=float) * hours_per_bin
    time_bin_end_h = time_bin_start_h + hours_per_bin
    footprint = footprint.assign_coords(
        time_ago=("time_ago", np.arange(n_time, dtype=int)),
        z_bin=("z_bin", z_centers_m),
        latitude=("latitude", lat_centers),
        longitude=("longitude", lon_centers),
    )

lon_min = float(lon_edges[0])
lon_max = float(lon_edges[-1])
lat_min = float(lat_edges[0])
lat_max = float(lat_edges[-1])

footprint.name = "footprint"
time_labels = [
    f"{start:g}-{end:g} h ago"
    for start, end in zip(time_bin_start_h, time_bin_end_h)
]

summary = pd.DataFrame(
    {
        "dimension": ["time_ago", "z_bin", "latitude", "longitude"],
        "size": [n_time, n_z, n_y, n_x],
        "min": [float(time_bin_start_h[0]), float(z_edges_m[0]), float(lat_edges[0]), float(lon_edges[0])],
        "max": [float(time_bin_end_h[-1]), float(z_edges_m[-1]), float(lat_edges[-1]), float(lon_edges[-1])],
    }
)
summary

,dimension,size,min,max
0,time_ago,10,0.0,10.0
1,z_bin,5,0.0,5000.0
2,latitude,16,35.9,39.9
3,longitude,16,-124.3,-120.3


## Quick dataset summary

In [3]:
nonzero_count = int(np.count_nonzero(np.nan_to_num(footprint.values, nan=0.0)))
total_cells = int(footprint.size)
peak_index = np.unravel_index(np.nanargmax(footprint.values), footprint.shape)
peak_value = float(footprint.values[peak_index])

stats = pd.DataFrame(
    [
        {"metric": "total footprint", "value": float(footprint.fillna(0.0).sum().item())},
        {"metric": "max cell", "value": peak_value},
        {"metric": "nonzero cells", "value": nonzero_count},
        {"metric": "nonzero share", "value": nonzero_count / total_cells},
    ]
)

peak_cell = pd.DataFrame(
    [
        {
            "time_bin": time_labels[peak_index[0]],
            "z_min_m": float(z_edges_m[peak_index[1]]),
            "z_max_m": float(z_edges_m[peak_index[1] + 1]),
            "latitude": float(footprint.latitude.values[peak_index[2]]),
            "longitude": float(footprint.longitude.values[peak_index[3]]),
            "value": peak_value,
        }
    ]
)

display(stats)
peak_cell

,metric,value
0,total footprint,29763.720703
1,max cell,1981.347656
2,nonzero cells,406.000000
3,nonzero share,0.031719


,time_bin,z_min_m,z_max_m,latitude,longitude,value
0,1-2 h ago,0.0,1000.0,37.775,-122.175,1981.347656


## Plot helpers

Build a per-cell GeoJSON over the footprint grid (used by the choropleth map below) and a small flattening helper that turns a `(time_ago, latitude, longitude)` slice into a long DataFrame for animation. Both are pure functions of the loaded coordinates so the rest of the notebook can stay declarative.

In [4]:
def build_grid_geojson() -> dict:
    """Build a FeatureCollection where each feature is one footprint grid cell."""
    features = []
    for y_idx in range(n_y):
        for x_idx in range(n_x):
            features.append(
                {
                    "type": "Feature",
                    "id": f"{y_idx}_{x_idx}",
                    "properties": {"y_idx": int(y_idx), "x_idx": int(x_idx)},
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [
                            [
                                [float(lon_edges[x_idx]), float(lat_edges[y_idx])],
                                [float(lon_edges[x_idx + 1]), float(lat_edges[y_idx])],
                                [float(lon_edges[x_idx + 1]), float(lat_edges[y_idx + 1])],
                                [float(lon_edges[x_idx]), float(lat_edges[y_idx + 1])],
                                [float(lon_edges[x_idx]), float(lat_edges[y_idx])],
                            ]
                        ],
                    },
                }
            )
    return {"type": "FeatureCollection", "features": features}


def footprint_long_dataframe(slice_3d: xr.DataArray) -> pd.DataFrame:
    """Flatten a (time_ago, latitude, longitude) DataArray into a long DataFrame.

    Drops zero/NaN cells so the choropleth doesn't draw an empty layer everywhere.
    """
    rows = []
    arr = np.nan_to_num(slice_3d.values, nan=0.0)
    for t_idx in range(arr.shape[0]):
        layer = arr[t_idx]
        for y_idx, x_idx in np.argwhere(layer > 0.0):
            value = float(layer[y_idx, x_idx])
            rows.append(
                {
                    "cell_id": f"{int(y_idx)}_{int(x_idx)}",
                    "time_ago": int(t_idx),
                    "time_label": time_labels[t_idx],
                    "longitude": float(lon_centers[x_idx]),
                    "latitude": float(lat_centers[y_idx]),
                    "footprint": value,
                    "log_footprint": float(np.log10(value)),
                }
            )
    return pd.DataFrame(rows)


def log_colorbar_ticks(vmin: float, vmax: float, *, max_ticks: int = 5) -> tuple[list[float], list[str]]:
    safe_vmin = max(vmin, 1e-12)
    safe_vmax = max(vmax, safe_vmin * 10.0) if vmax <= vmin else vmax
    n_ticks = min(max_ticks, max(2, int(np.ceil(np.log10(safe_vmax) - np.log10(safe_vmin))) + 1))
    tickvals = np.linspace(np.log10(safe_vmin), np.log10(safe_vmax), num=n_ticks)
    ticktext = [f"{10 ** v:.2g}" for v in tickvals]
    return tickvals.tolist(), ticktext


grid_geojson = build_grid_geojson()
print(f"Built {len(grid_geojson['features'])} grid-cell features.")

Built 256 grid-cell features.


## Animated map: footprint at the lowest level

Footprint cells colored by log₁₀(value) at one altitude bin, animated through the time-ago bins. Press play, scrub the slider, hover for cell values, and zoom/pan to interrogate. Change `LEVEL_INDEX` and rerun to view a different altitude bin.

In [5]:
LEVEL_INDEX = 0
level_label = f"{z_edges_m[LEVEL_INDEX]:.0f}-{z_edges_m[LEVEL_INDEX + 1]:.0f} m AGL"

slice_at_level = footprint.isel(z_bin=LEVEL_INDEX)
df = footprint_long_dataframe(slice_at_level)

if df.empty:
    print("No nonzero footprint cells at this level - try a different LEVEL_INDEX.")
    fig = None
else:
    df = df.sort_values("time_ago").reset_index(drop=True)
    vmin = float(df["footprint"].min())
    vmax = float(df["footprint"].max())
    log_min = np.log10(max(vmin, 1e-12))
    log_max = np.log10(vmax) if vmax > vmin else log_min + 1.0
    tickvals, ticktext = log_colorbar_ticks(vmin, vmax)

    span = max(float(lon_max - lon_min), float(lat_max - lat_min))
    zoom = float(np.clip(8.0 - np.log2(max(span, 0.25)), 4.0, 9.0))

    fig = px.choropleth_map(
        df,
        geojson=grid_geojson,
        locations="cell_id",
        color="log_footprint",
        animation_frame="time_ago",
        hover_data={
            "time_label": True,
            "footprint": ":.3g",
            "longitude": ":.3f",
            "latitude": ":.3f",
            "log_footprint": False,
            "cell_id": False,
            "time_ago": False,
        },
        map_style="carto-positron",
        center={"lat": release_lat, "lon": release_lon},
        zoom=zoom,
        opacity=0.75,
        color_continuous_scale="Magma",
        range_color=[log_min, log_max],
        labels={"log_footprint": "log10(footprint)"},
        title=f"Footprint at {level_label}",
    )

    fig.add_trace(
        go.Scattermap(
            lon=[release_lon],
            lat=[release_lat],
            mode="markers",
            marker={"size": 12, "color": "cyan"},
            name="Release",
            showlegend=False,
            hovertemplate="Release<br>Lon: %{lon:.3f}<br>Lat: %{lat:.3f}<extra></extra>",
        )
    )

    fig.update_coloraxes(colorbar={"tickvals": tickvals, "ticktext": ticktext, "len": 0.85})

    if fig.layout.sliders:
        slider = fig.layout.sliders[0]
        slider.currentvalue = {"prefix": "Time: "}
        for step in slider.steps:
            try:
                idx = int(step.label)
            except (TypeError, ValueError):
                continue
            if 0 <= idx < len(time_labels):
                step.label = time_labels[idx]

    fig.update_layout(
        height=520,
        margin={"l": 10, "r": 10, "t": 50, "b": 10},
    )

fig

## Time-altitude profile (horizontally integrated)

Heatmap of footprint summed across all latitude/longitude cells, as a function of time before release and altitude bin. Useful for diagnosing when and at what altitude particles spend their time in the sensitivity-relevant region.

In [6]:
ta_profile = footprint.sum(dim=["latitude", "longitude"]).values  # (T, Z)
positive_mask = ta_profile > 0.0

if not positive_mask.any():
    print("Footprint is empty - nothing to render in time-altitude profile.")
    fig = None
else:
    vmin = float(ta_profile[positive_mask].min())
    vmax = float(ta_profile[positive_mask].max())
    log_min = np.log10(max(vmin, 1e-12))
    log_max = np.log10(vmax) if vmax > vmin else log_min + 1.0
    tickvals, ticktext = log_colorbar_ticks(vmin, vmax)

    log_values = np.where(positive_mask, np.log10(np.maximum(ta_profile, 1e-12)), np.nan)

    fig = go.Figure(
        go.Heatmap(
            z=log_values.T,
            x=time_labels,
            y=z_centers_m,
            colorscale="Magma",
            zmin=log_min,
            zmax=log_max,
            customdata=ta_profile.T,
            colorbar={
                "title": "log10(footprint)",
                "tickvals": tickvals,
                "ticktext": ticktext,
                "len": 0.9,
            },
            hovertemplate=(
                "Time: %{x}<br>Altitude: %{y:.0f} m AGL<br>Footprint: %{customdata:.3g}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        title="Footprint vs time and altitude (horizontally integrated)",
        xaxis_title="Time before release",
        yaxis_title="Altitude AGL (m)",
        height=380,
        margin={"l": 70, "r": 10, "t": 50, "b": 60},
    )

fig

## Total footprint by time before release

Footprint integrated horizontally and split by altitude bin, on a log y-axis. The black line is the all-levels total; colored lines show per-level contributions. Useful for spotting when particles enter/leave the receptor's sensitivity window.

In [7]:
total_by_level_time = footprint.sum(dim=["latitude", "longitude"]).values  # (T, Z)
total_by_time = total_by_level_time.sum(axis=1)  # (T,)

x_hours = 0.5 * (np.asarray(time_bin_start_h) + np.asarray(time_bin_end_h))

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_hours,
        y=total_by_time,
        mode="lines+markers",
        name="Total (all levels)",
        line={"color": "black", "width": 2.5},
    )
)
for z_idx in range(n_z):
    label = f"{z_edges_m[z_idx]:.0f}-{z_edges_m[z_idx + 1]:.0f} m"
    fig.add_trace(
        go.Scatter(
            x=x_hours,
            y=total_by_level_time[:, z_idx],
            mode="lines",
            name=label,
            line={"width": 1.4},
        )
    )

fig.update_layout(
    title="Total footprint by time before release",
    xaxis_title="Time before release (hours, bin midpoint)",
    yaxis_title="Footprint",
    yaxis_type="log",
    height=380,
    margin={"l": 60, "r": 10, "t": 50, "b": 60},
    legend={"orientation": "v", "yanchor": "top", "y": 1.0, "xanchor": "left", "x": 1.02},
)
fig

## Raw dataset handle

Keep this at the end if you want to inspect the reconstructed `xarray.DataArray` directly.

In [ ]:
footprint